# FootyPredict: Dual-Stage Machine Learning Framework for Football Match Forecasting
### Course: Artificial Intelligence & Data Science II (AI & DS II)
**Student:** Saad Vakharia &nbsp;|&nbsp; **Roll No:** 155 &nbsp;|&nbsp; **Class:** BEIT-2 &nbsp;|&nbsp; **Batch:** B2  
**Institution:** Atharva College of Engineering, University of Mumbai  
**Faculty Guide:** Prof. Suvarna Panchal  

---
## 1. Problem Definition & Objectives
Predicting association football match outcomes is fundamentally challenging due to high match entropy and low scoring events. This project introduces a **Dual-Stage Machine Learning Architecture**:
1. **Stage 1 (Classification):** Discrete Match Result prediction ($H$: Home Win, $D$: Draw, $A$: Away Win) utilizing Gradient Boosted Trees, Random Forest, Logistic Regression, and Neural Networks.
2. **Stage 2 (Poisson Modeling):** Bivariate Poisson regression predicting Expected Goals ($xG$) and generating the full joint scoreline probability distribution matrix $P(X=x, Y=y)$.

In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Verify working directory and load dataset
data_path = os.path.join('..', 'data', 'matches.csv')
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} historical European match records.")
df.head()

## 2. Exploratory Data Analysis (EDA)
We inspect class distribution across Home Wins, Draws, and Away Wins, as well as the goal-scoring distributions.

In [2]:
outcome_counts = df['FTR'].value_counts(normalize=True) * 100
print("Match Result Distribution (%):")
print(outcome_counts.round(2))

print(f"\nAverage Home Goals: {df['FTHG'].mean():.2f}")
print(f"Average Away Goals: {df['FTAG'].mean():.2f}")
print(f"Average Elo Differential: {df['EloDiff'].mean():.1f} (std: {df['EloDiff'].std():.1f})")

## 3. Feature Preparation & Preprocessing
We isolate engineered pre-match indicators (Elo differential, rolling 5-match form points, goal differential momentum, rest days, and match tempo stats) and perform standardized 80-20 train-test splitting.

In [3]:
from src.data_loader import prepare_features, split_and_scale

X, y_outcome, y_hg, y_ag = prepare_features(df)
data = split_and_scale(X, y_outcome, y_hg, y_ag, test_size=0.2, random_state=42)

print(f"Training set: {data['X_train'].shape[0]} samples")
print(f"Testing set:  {data['X_test'].shape[0]} samples")

## 4. Stage 1: Multi-Model Outcome Classification Benchmarking
Evaluating models on mandatory academic assessment metrics: **Accuracy, Precision (macro), Recall (macro), and F1-Score (macro & weighted)**.

In [4]:
from src.model import benchmark_all_classifiers

bench_results = benchmark_all_classifiers(data['X_train'], data['y_out_train'], data['X_test'], data['y_out_test'])
bench_df = pd.DataFrame(bench_results).T[['Accuracy', 'Precision_Macro', 'Recall_Macro', 'F1_Macro', 'F1_Weighted']]
bench_df

## 5. Stage 2: Bivariate Poisson Modeling & Dual-Stage Ensemble
We fit Poisson goal expectancy regressors $\hat{\lambda}_h$ and $\hat{\lambda}_a$ to generate exact scoreline probability matrices, and blend the probabilities with the Stage 1 Classifier.

In [5]:
from src.model import DualStageFootballPredictor

dual_model = DualStageFootballPredictor(classifier_type="gradient_boosting")
dual_model.fit(data['X_train'], data['y_out_train'], data['y_hg_train'], data['y_ag_train'])

# Predict single test fixture
sample_x = data['X_test'][0]
match_pred = dual_model.predict_match(sample_x)

print("Outcome Probabilities:", match_pred['outcome_probs'])
print("Expected Goals (xG):", match_pred['expected_goals'])
print("Top 5 Scorelines:")
for sc in match_pred['top_scorelines']:
    print(f"  {sc['score']:<6} -> Probability: {sc['probability']*100:.2f}%")

## 6. Evaluation Visualizations & Performance Summary
We inspect the confusion matrix and feature importance ranking for our dual-stage framework.

In [6]:
from src.evaluate import run_evaluation_suite
run_evaluation_suite()
print("Evaluation suite completed. Figures saved to report/figures/ directory.")